Complete training loop with best practices

Production-ready loop with validation, LR scheduling, gradient clipping, and checkpointing




In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR


# setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

# optional -> torch.compile() for 2-3x speed

model = torch.compile(model)

# Learning Rate Scheule: warmup + cosine annealing
total_epochs = 50
warmup_epochs = 5
warmup = LinearLR(optimizer, start_factor=0.1, total_iters = warmup_epochs)
cosine = CosineAnnealingLR(optimizer, T_max = total_epochs - warmup_epochs)
scheduler = SequentialLR(optimizer, [warmup,cosine], milestones=[warmup_epochs])


from torch.utils.data import DataLoader, TensorDataset

train_data = TensorDataset(torch.randn(640, 784), torch.randint(0, 10, (640,)))
val_data   = TensorDataset(torch.randn(160, 784), torch.randint(0, 10, (160,)))
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32)

best_val_loss = float('inf')
patience, patience_counter = 10, 0

for epoch in range(total_epochs):
  model.train()
  train_loss = 0.0
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()

    # gradient clipping before optimizer step
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    train_loss += loss.item()

  model.eval()
  val_loss, correct, total = 0.0, 0, 0 # Initialize val_loss inside the epoch loop

  with torch.no_grad():
    for inputs, targets in val_loader:
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)
      val_loss += criterion(outputs, targets).item()
      _, predicted = outputs.max(1)
      total += targets.size(0)
      correct += predicted.eq(targets).sum().item()

  avg_train = train_loss / len(train_loader)
  avg_val = val_loss / len(val_loader)
  accuracy = 100. * correct / total

  scheduler.step()

  print(f"Epoch {epoch+1}/{total_epochs} | "
    f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
    f"Acc: {accuracy:.1f}% | LR: {scheduler.get_last_lr()[0]:.6f}")

  if avg_val < best_val_loss:
          best_val_loss = avg_val
          patience_counter = 0
          torch.save({
              'epoch': epoch,
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict(),
              'scheduler_state_dict': scheduler.state_dict(),
              'best_val_loss': best_val_loss,
          }, 'best_model.pt')
          print(f"  -> Saved best model (val_loss: {best_val_loss:.4f})")
  else:
          patience_counter += 1
          if patience_counter >= patience:
              print(f"Early stopping at epoch {epoch+1}")
              break

# Load best model for evaluation
checkpoint = torch.load('best_model.pt', weights_only=False)  # weights_only=False needed for optimizer/scheduler state
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

Epoch 1/50 | Train: 2.3042 | Val: 2.3096 | Acc: 13.1% | LR: 0.000084
  -> Saved best model (val_loss: 2.3096)
Epoch 2/50 | Train: 2.2884 | Val: 2.3110 | Acc: 11.2% | LR: 0.000138
Epoch 3/50 | Train: 2.2667 | Val: 2.3115 | Acc: 10.0% | LR: 0.000192
Epoch 4/50 | Train: 2.2229 | Val: 2.3149 | Acc: 8.8% | LR: 0.000246
Epoch 5/50 | Train: 2.1728 | Val: 2.3183 | Acc: 7.5% | LR: 0.000300
Epoch 6/50 | Train: 2.0806 | Val: 2.3292 | Acc: 6.9% | LR: 0.000300
Epoch 7/50 | Train: 1.9497 | Val: 2.3351 | Acc: 8.1% | LR: 0.000299
Epoch 8/50 | Train: 1.7890 | Val: 2.3419 | Acc: 10.0% | LR: 0.000297
Epoch 9/50 | Train: 1.5910 | Val: 2.3590 | Acc: 10.0% | LR: 0.000294
Epoch 10/50 | Train: 1.3519 | Val: 2.3720 | Acc: 9.4% | LR: 0.000291
Epoch 11/50 | Train: 1.0984 | Val: 2.3855 | Acc: 8.8% | LR: 0.000287
Early stopping at epoch 11
Loaded best model from epoch 1


Mixed precision training with AMP (automatic mixed precision)

Train with float16/bfloat16 for ~2x speedup and ~40% memory savings




In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# self contained and can run standalone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Linear(784,256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128,10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
num_epochs = 3

# syntheic data for demonstration
train_data = TensorDataset(torch.randn(320,784), torch.randint(0,10, (320,)))
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# GradScaler prevents gradients underflow in float16
# not needed for bfloat16 (use autocast alone)
use_amp = device.type == 'cuda'
scaler = GradScaler('cuda') if use_amp else None

# training with fp16 (requries grad scaler)

for epoch in range(num_epochs):
  model.train()
  for inputs, targets in train_loader:
    inputs = inputs.to(device)
    targets = targets.to(device)

    optimizer.zero_grad()

    # autocast run forward pass in float16 where safe
    with autocast('cuda', enabled=use_amp):
      outputs = model(inputs)
      loss = criterion(outputs, targets)

    if use_amp:
      scaler.scale(loss).backward()
      scaler.unscale_(optimizer)
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0)
      scaler.step(optimizer)
      scaler.update()
    else:
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()

# bfloat16 simpler and no scaler needed
# Preferred on Ampere+ GPUs (A100, H100, RTX 3090+)
for inputs, targets in train_loader:
  inputs = inputs.to(device)
  targets = targets.to(device)

  optimizer.zero_grad()

  # bfloat16 has sae exponent range as float32
  # no loss scaling needed
  with autocast('cuda', dtype=torch.bfloat16):
    outputs = model(inputs)
    loss = criterion(outputs, targets)

  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
  optimizer.step()



# memory comparioson
def count_memory():
  allocated = torch.cuda.memory_allocated() / 1e9
  reserved = torch.cuda.memory_reserved() / 1e9
  return f"Allocated: {allocated:4f} and Reserved: {reserved:4f}"

print(f"Memory Usage: {count_memory()}")
print(f"Typical Savings: ~40-60% less memory with mixed precision")

Memory Usage: Allocated: 0.000000 and Reserved: 0.000000
Typical Savings: ~40-60% less memory with mixed precision


/tmp/ipykernel_768/767353018.py:65: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with autocast('cuda', dtype=torch.bfloat16):


Gradient accumulation for large effective batches

Simulate batch_size=256 when GPU only fits batch_size=32

In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ============================================================
# SETUP (self-contained - can run standalone)
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = nn.Sequential(
    nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 10)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 3
accumulation_steps = 8  # effective batch = 32 * 8 = 256

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    total_steps_in_epoch = len(train_loader)

    for step, (inputs, targets) in enumerate(train_loader):
        inputs = inputs.to(device)
        targets = targets.to(device)

        # 1. Forward pass
        outputs = model(inputs)

        # 2. Divide loss by accumulation steps
        loss = criterion(outputs, targets) / accumulation_steps

        # 3. Backward pass (accumulates gradients)
        loss.backward()

        # 4. Update weights every accumulation_steps
        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()  # Step scheduler on effective batch update

            effective_step = (step + 1) // accumulation_steps
    print(f"Epoch {epoch+1} | Step {effective_step} | Loss: {loss.item() * accumulation_steps:.4f}")

    # 5. Handle remaining gradients at epoch end (if total batches don't divide evenly)
    if total_steps_in_epoch % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()

# ============================================================
# KEY POINTS
# ============================================================
# 1. Divide loss by accumulation_steps (keeps gradient scale correct)
# 2. Only step optimizer every N mini-batches
# 3. Zero gradients AFTER optimizer step, not before backward
# 4. Handle partial accumulation at epoch boundaries
# 5. Gradient clipping applies to the ACCUMULATED gradients

Epoch 1 | Step 2 | Loss: 2.2949
Epoch 2 | Step 2 | Loss: 2.2767
Epoch 3 | Step 2 | Loss: 2.2684
